# 月初 / 月末贡献稳定性诊断

目标：解释按 weekday 对齐后，月初和月末贡献占比为什么不稳定。

这份 notebook 面向业务老板和产品老板，尽量少用统计术语，重点回答三个问题：

1. 月末是否真的有“冲量”，还是只是工作日少导致贡献占比自然变大？
2. 当月工作日数、月末/次月初节假日，是否会影响月末贡献？
3. 哪些月份需要业务重点复盘？

数据源：`data/sales_30d_daily.csv`，字段为 `bizym / transdate / num_hosp / qty`。


## 1. 准备数据和统一口径

口径说明：

- 用 `chinese_calendar` 识别中国工作日；如果本地环境没有该包，则自动退化为周一到周五。
- “贡献占比” = 当天销量 / 当月销量。
- “月末冲量”优先看最后 5 个工作日相对月中日均的提升，而不是只看最后 5 天占比。这样可以减少“当月工作日少，最后 5 天天然占比高”的误判。


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings("ignore")

def find_repo_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'code').is_dir() and (candidate / 'data').is_dir():
            return candidate
    raise FileNotFoundError("Cannot locate repository root containing code/ and data/")

REPO_ROOT = find_repo_root()
DATA_PATH = REPO_ROOT / 'data' / 'sales_30d_daily.csv'

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_theme(style="whitegrid", font="sans-serif")
plt.rcParams.update({
    "figure.dpi": 140,
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "legend.fontsize": 10,
    "axes.unicode_minus": False,
    "font.sans-serif": [
        "Arial Unicode MS", "PingFang SC", "Hiragino Sans GB", "Microsoft YaHei",
        "SimHei", "Noto Sans CJK SC", "DejaVu Sans"
    ],
})

PALETTE = {
    "blue": "#4C72B0",
    "orange": "#DD8452",
    "green": "#55A868",
    "red": "#C44E52",
    "purple": "#8172B3",
    "gray": "#8C8C8C",
    "light_gray": "#D9D9D9",
}

def fmt_pct(x, digits=1):
    if pd.isna(x):
        return ""
    return f"{x * 100:.{digits}f}%"

def fmt_num(x):
    if pd.isna(x):
        return ""
    return f"{x:,.0f}"

def pct_axis(ax):
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0, decimals=0))

try:
    from chinese_calendar import is_workday as cn_is_workday, is_holiday as cn_is_holiday
    CALENDAR_SOURCE = "chinese_calendar"
    def is_workday(ts):
        return bool(cn_is_workday(pd.Timestamp(ts).date()))
    def is_holiday(ts):
        return bool(cn_is_holiday(pd.Timestamp(ts).date()))
except Exception as exc:
    CALENDAR_SOURCE = f"weekday_fallback_mon_fri ({exc})"
    def is_workday(ts):
        return pd.Timestamp(ts).dayofweek < 5
    def is_holiday(ts):
        return not is_workday(ts)

print(f"calendar_source = {CALENDAR_SOURCE}")


In [ ]:
daily_raw = pd.read_csv(DATA_PATH)
daily_raw["transdate"] = pd.to_datetime(daily_raw["transdate"])
daily_raw = daily_raw.sort_values(["bizym", "transdate"]).reset_index(drop=True)

daily = daily_raw.copy()
daily["year"] = daily["transdate"].dt.year
daily["month"] = daily["transdate"].dt.month
daily["month_start"] = daily["transdate"].dt.to_period("M").dt.to_timestamp()
daily["month_label"] = daily["month_start"].dt.strftime("%Y-%m")
daily["day_of_month"] = daily["transdate"].dt.day
daily["weekday"] = daily["transdate"].dt.dayofweek + 1

daily["is_workday"] = daily["transdate"].map(is_workday).astype(bool)
daily["is_holiday"] = daily["transdate"].map(is_holiday).astype(bool)

daily["month_total_qty"] = daily.groupby("bizym")["qty"].transform("sum")
daily["daily_contrib"] = daily["qty"] / daily["month_total_qty"]
daily["workday_seq"] = (
    daily.groupby("bizym")["is_workday"]
    .cumsum()
    .where(daily["is_workday"], np.nan)
)
daily["month_workdays"] = daily.groupby("bizym")["is_workday"].transform("sum")
daily["workdays_to_month_end"] = daily["month_workdays"] - daily["workday_seq"]

coverage = {
    "start_date": daily["transdate"].min().date(),
    "end_date": daily["transdate"].max().date(),
    "rows": len(daily),
    "months": daily["bizym"].nunique(),
}
coverage


## 2. 定义老板能理解的三个指标

为了避免“最后 5 个工作日占比高”被误读，这里同时看三个指标：

- **末 5 工作日贡献**：最后 5 个工作日贡献占整月多少。
- **末 5 理论基准**：如果每天均匀出货，最后 5 个工作日理论上应贡献 `5 / 当月工作日数`。
- **月末提升**：最后 5 个工作日日均贡献，相比月中工作日日均贡献高多少。这个更接近真实“冲量”。


In [ ]:
def holiday_count(dates):
    return int(sum(is_holiday(d) for d in dates))

def build_month_panel(daily_df):
    rows = []
    for bizym, g in daily_df.groupby("bizym"):
        g = g.sort_values("transdate").copy()
        y = int(bizym) // 100
        m = int(bizym) % 100
        month_start = pd.Timestamp(y, m, 1)
        month_end = month_start + pd.offsets.MonthEnd(0)
        next_month_start = month_end + pd.Timedelta(days=1)
        next_7 = pd.date_range(next_month_start, periods=7, freq="D")
        month_last_7 = pd.date_range(max(month_start, month_end - pd.Timedelta(days=6)), month_end, freq="D")
        month_first_7 = pd.date_range(month_start, periods=min(7, month_end.day), freq="D")

        wg = g[g["is_workday"]].copy()
        total_qty = g["qty"].sum()
        workdays = int(wg["transdate"].nunique())
        if workdays == 0 or total_qty == 0:
            continue

        first5 = wg.head(5)
        last5 = wg.tail(5)
        middle = wg.iloc[5:-5] if len(wg) > 10 else wg.iloc[0:0]

        first5_contrib = first5["qty"].sum() / total_qty
        last5_contrib = last5["qty"].sum() / total_qty
        first3_contrib = wg.head(3)["qty"].sum() / total_qty
        last3_contrib = wg.tail(3)["qty"].sum() / total_qty
        mid_avg_contrib = middle["daily_contrib"].mean()
        last5_avg_contrib = last5["daily_contrib"].mean()
        first5_avg_contrib = first5["daily_contrib"].mean()

        rows.append({
            "bizym": int(bizym),
            "year": y,
            "month": m,
            "month_start": month_start,
            "month_label": month_start.strftime("%Y-%m"),
            "total_qty": total_qty,
            "workdays": workdays,
            "calendar_days": int(g["transdate"].nunique()),
            "first_weekday": int(month_start.dayofweek + 1),
            "last_weekday": int(month_end.dayofweek + 1),
            "first5_contrib": first5_contrib,
            "last5_contrib": last5_contrib,
            "first3_contrib": first3_contrib,
            "last3_contrib": last3_contrib,
            "even_last5_baseline": min(5, workdays) / workdays,
            "last5_excess_vs_even": last5_contrib - min(5, workdays) / workdays,
            "end_minus_start_5": last5_contrib - first5_contrib,
            "mid_avg_daily_contrib": mid_avg_contrib,
            "last5_avg_daily_contrib": last5_avg_contrib,
            "first5_avg_daily_contrib": first5_avg_contrib,
            "end_lift_vs_mid": last5_avg_contrib / mid_avg_contrib - 1 if pd.notna(mid_avg_contrib) and mid_avg_contrib > 0 else np.nan,
            "start_lift_vs_mid": first5_avg_contrib / mid_avg_contrib - 1 if pd.notna(mid_avg_contrib) and mid_avg_contrib > 0 else np.nan,
            "month_first_7_holidays": holiday_count(month_first_7),
            "month_last_7_holidays": holiday_count(month_last_7),
            "next_month_first_3_holidays": holiday_count(next_7[:3]),
            "next_month_first_5_holidays": holiday_count(next_7[:5]),
            "next_month_first_7_holidays": holiday_count(next_7),
        })
    return pd.DataFrame(rows).sort_values("month_start").reset_index(drop=True)

month_panel = build_month_panel(daily)
month_panel["next_first5_contrib"] = month_panel["first5_contrib"].shift(-1)
month_panel["next_month_label"] = month_panel["month_label"].shift(-1)

summary_view = month_panel[[
    "month_label", "total_qty", "workdays", "last5_contrib", "even_last5_baseline",
    "last5_excess_vs_even", "end_lift_vs_mid", "next_month_first_5_holidays"
]].tail(12).copy()

summary_view["total_qty"] = summary_view["total_qty"].map(fmt_num)
for col in ["last5_contrib", "even_last5_baseline", "last5_excess_vs_even", "end_lift_vs_mid"]:
    summary_view[col] = summary_view[col].map(fmt_pct)
summary_view.rename(columns={
    "month_label": "月份",
    "total_qty": "月销量",
    "workdays": "工作日数",
    "last5_contrib": "末5工作日贡献",
    "even_last5_baseline": "均匀出货基准",
    "last5_excess_vs_even": "高于基准",
    "end_lift_vs_mid": "月末相对月中提升",
    "next_month_first_5_holidays": "次月前5天假日数",
})


## 3. 一眼看懂：最近月份的月末贡献有没有异常？

这张图同时放了实际贡献和“均匀出货基准”。

- 蓝线：最后 5 个工作日实际贡献。
- 灰线：如果工作日均匀出货，最后 5 个工作日理论贡献。
- 橙色柱：实际贡献高于理论基准的部分，越高越值得复盘。


In [ ]:
plot_df = month_panel.tail(24).copy()

fig, ax = plt.subplots(figsize=(13, 5.5))
ax.plot(plot_df["month_label"], plot_df["last5_contrib"], marker="o", linewidth=2.2,
        color=PALETTE["blue"], label="末5工作日实际贡献")
ax.plot(plot_df["month_label"], plot_df["even_last5_baseline"], marker="o", linewidth=1.8,
        color=PALETTE["gray"], linestyle="--", label="均匀出货基准")

positive = plot_df["last5_excess_vs_even"].clip(lower=0)
ax.bar(plot_df["month_label"], positive, bottom=plot_df["even_last5_baseline"],
       color=PALETTE["orange"], alpha=0.35, label="高于基准的部分")

for _, row in plot_df.nlargest(3, "last5_excess_vs_even").iterrows():
    ax.annotate(
        row["month_label"],
        (row["month_label"], row["last5_contrib"]),
        textcoords="offset points", xytext=(0, 9), ha="center", fontsize=9,
        color=PALETTE["orange"]
    )

pct_axis(ax)
ax.set_title("最近24个月：末5工作日贡献 vs 均匀出货基准")
ax.set_xlabel("")
ax.set_ylabel("占当月销量")
ax.tick_params(axis="x", rotation=45)
ax.legend(loc="upper left", frameon=True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()


## 4. 月末冲量热力图：哪些月份反复不稳定？

这里看“月末相对月中提升”。颜色越深，代表最后 5 个工作日日均贡献越高于月中。

这张图适合回答：是某些自然月份一直容易月末冲量，还是只有个别年份异常？


In [ ]:
heat = month_panel.pivot(index="year", columns="month", values="end_lift_vs_mid")

fig, ax = plt.subplots(figsize=(12, 4.8))
sns.heatmap(
    heat,
    ax=ax,
    cmap="YlOrRd",
    center=0,
    annot=heat.map(lambda x: "" if pd.isna(x) else f"{x:.0%}"),
    fmt="",
    linewidths=0.6,
    linecolor="white",
    cbar_kws={"label": "月末相对月中提升"},
)
ax.set_title("月末相对月中提升：按年份 x 月份查看")
ax.set_xlabel("月份")
ax.set_ylabel("年份")
ax.set_xticklabels([f"{int(x.get_text()):02d}" for x in ax.get_xticklabels()], rotation=0)
plt.tight_layout()
plt.show()


## 5. 工作日少，会不会天然放大月末贡献？

这张图不追求复杂统计，只看业务直觉：

- 横轴：当月工作日数。
- 纵轴：末 5 工作日贡献。
- 点越大、颜色越深：次月前 5 天假日越多。

如果工作日少，最后 5 个工作日占比本来就会更高，所以要结合上一张“相对月中提升”一起判断。


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(
    month_panel["workdays"],
    month_panel["last5_contrib"],
    s=70 + month_panel["next_month_first_5_holidays"] * 35,
    c=month_panel["next_month_first_5_holidays"],
    cmap="YlOrRd",
    alpha=0.82,
    edgecolor="white",
    linewidth=0.8,
)

for _, row in month_panel.nlargest(6, "last5_contrib").iterrows():
    ax.annotate(row["month_label"], (row["workdays"], row["last5_contrib"]),
                textcoords="offset points", xytext=(5, 4), fontsize=9)

ax.set_title("工作日数 vs 末5工作日贡献")
ax.set_xlabel("当月工作日数")
ax.set_ylabel("末5工作日贡献")
pct_axis(ax)
cb = plt.colorbar(scatter, ax=ax)
cb.set_label("次月前5天假日数")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()


## 6. 次月初是假期，会不会把销量提前到本月末？

如果“次月初休假导致本月末提前备货/冲量”成立，通常会看到：

- 当月末 5 工作日贡献偏高；
- 次月初 5 工作日贡献偏低；
- 点会落在右下角。

下面这张图直接检查这个业务故事是否明显。


In [ ]:
cross_month = month_panel.dropna(subset=["next_first5_contrib"]).copy()

fig, ax = plt.subplots(figsize=(9.5, 6))
scatter = ax.scatter(
    cross_month["last5_contrib"],
    cross_month["next_first5_contrib"],
    s=75 + cross_month["next_month_first_5_holidays"] * 35,
    c=cross_month["next_month_first_5_holidays"],
    cmap="YlOrRd",
    alpha=0.82,
    edgecolor="white",
    linewidth=0.8,
)

candidate = cross_month.assign(score=cross_month["last5_contrib"] - cross_month["next_first5_contrib"]).nlargest(6, "score")
for _, row in candidate.iterrows():
    ax.annotate(f"{row['month_label']} -> {row['next_month_label']}",
                (row["last5_contrib"], row["next_first5_contrib"]),
                textcoords="offset points", xytext=(5, 4), fontsize=9)

ax.axvline(cross_month["last5_contrib"].median(), color=PALETTE["light_gray"], linestyle="--", linewidth=1)
ax.axhline(cross_month["next_first5_contrib"].median(), color=PALETTE["light_gray"], linestyle="--", linewidth=1)
ax.set_title("当月末5贡献 vs 次月初5贡献：是否存在提前冲量？")
ax.set_xlabel("当月末5工作日贡献")
ax.set_ylabel("次月初5工作日贡献")
ax.xaxis.set_major_formatter(mticker.PercentFormatter(1.0, decimals=0))
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0, decimals=0))
cb = plt.colorbar(scatter, ax=ax)
cb.set_label("次月前5天假日数")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()


## 7. 典型月份拆解：到底是连续冲量，还是单日异常？

下面挑选“月末相对月中提升”最高的月份，画日级贡献曲线。

业务复盘时重点看两件事：

- 是最后一周连续变高，还是只有最后一天特别高？
- 月初是否也异常高？如果月初和月末都高，可能是节假日/月历结构，而不一定是纯月末冲量。


In [ ]:
top_months = month_panel.nlargest(6, "end_lift_vs_mid")["bizym"].tolist()
case_df = daily[daily["bizym"].isin(top_months) & daily["is_workday"]].copy()
case_df["workday_seq"] = case_df["workday_seq"].astype(int)

n = len(top_months)
fig, axes = plt.subplots(2, 3, figsize=(15, 7.5), sharey=True)
axes = axes.flatten()

for ax, bizym in zip(axes, top_months):
    g = case_df[case_df["bizym"] == bizym].sort_values("workday_seq")
    info = month_panel.loc[month_panel["bizym"] == bizym].iloc[0]
    ax.plot(g["workday_seq"], g["daily_contrib"], marker="o", linewidth=2, color=PALETTE["blue"])
    ax.axvspan(max(g["workday_seq"].max() - 4, 1), g["workday_seq"].max(), color=PALETTE["orange"], alpha=0.15)
    ax.set_title(f"{info['month_label']} | 月末提升 {info['end_lift_vs_mid']:.0%}")
    ax.set_xlabel("工作日序号")
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0, decimals=0))
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

for ax in axes[n:]:
    ax.set_visible(False)
axes[0].set_ylabel("单日贡献")
fig.suptitle("月末提升最高的典型月份：日级贡献曲线", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


## 8. 复盘清单：优先看哪些月份？

这里给出两类月份：

- **可能月末冲量**：月末相对月中提升高，且末 5 工作日贡献高于均匀基准。
- **可能跨月提前**：当月末 5 工作日贡献高，同时次月初 5 工作日贡献低。

这些月份适合拿给销售、运营、产品一起复盘：是否有政策截止、渠道压货、节前备货、开票/考核节点等原因。


In [ ]:
review = month_panel.copy()
review["end_push_score"] = review["end_lift_vs_mid"].fillna(0) + review["last5_excess_vs_even"].fillna(0)
review["cross_month_score"] = review["last5_contrib"] - review["next_first5_contrib"]

review_table = review.sort_values("end_push_score", ascending=False).head(12)[[
    "month_label", "total_qty", "workdays", "last5_contrib", "even_last5_baseline",
    "last5_excess_vs_even", "end_lift_vs_mid", "next_month_first_5_holidays", "next_first5_contrib"
]].copy()

review_table["total_qty"] = review_table["total_qty"].map(fmt_num)
for col in ["last5_contrib", "even_last5_baseline", "last5_excess_vs_even", "end_lift_vs_mid", "next_first5_contrib"]:
    review_table[col] = review_table[col].map(fmt_pct)

review_table.rename(columns={
    "month_label": "月份",
    "total_qty": "月销量",
    "workdays": "工作日数",
    "last5_contrib": "末5工作日贡献",
    "even_last5_baseline": "均匀基准",
    "last5_excess_vs_even": "高于基准",
    "end_lift_vs_mid": "月末相对月中提升",
    "next_month_first_5_holidays": "次月前5天假日数",
    "next_first5_contrib": "次月初5工作日贡献",
})


## 9. 自动生成一句话结论

这部分用于快速形成会议里的口径，后续可以结合业务事件再补充解释。


In [ ]:
latest = month_panel.iloc[-1]
top_end = month_panel.nlargest(3, "end_lift_vs_mid")
top_excess = month_panel.nlargest(3, "last5_excess_vs_even")
cross_candidates = cross_month.assign(score=cross_month["last5_contrib"] - cross_month["next_first5_contrib"]).nlargest(3, "score")

print(f"数据覆盖 {daily['transdate'].min().date()} 至 {daily['transdate'].max().date()}，共 {month_panel['bizym'].nunique()} 个自然月。")
print()
print("月末相对月中提升最高的月份：")
for _, row in top_end.iterrows():
    print(f"- {row['month_label']}：月末相对月中提升 {fmt_pct(row['end_lift_vs_mid'])}，末5工作日贡献 {fmt_pct(row['last5_contrib'])}，工作日数 {int(row['workdays'])}。")
print()
print("末5工作日贡献高于均匀基准最多的月份：")
for _, row in top_excess.iterrows():
    print(f"- {row['month_label']}：末5贡献 {fmt_pct(row['last5_contrib'])}，均匀基准 {fmt_pct(row['even_last5_baseline'])}，高出 {fmt_pct(row['last5_excess_vs_even'])}。")
print()
print("可能存在跨月提前的月份：")
for _, row in cross_candidates.iterrows():
    print(f"- {row['month_label']} -> {row['next_month_label']}：当月末5贡献 {fmt_pct(row['last5_contrib'])}，次月初5贡献 {fmt_pct(row['next_first5_contrib'])}，次月前5天假日数 {int(row['next_month_first_5_holidays'])}。")
print()
print("业务解读建议：")
print("- 如果某月只是在末5贡献高，但没有高于均匀基准很多，优先解释为工作日数量造成的自然占比变化。")
print("- 如果某月同时满足：月末相对月中提升高、末5贡献高于基准、次月初贡献偏低，再进入'节前提前/月底冲量'复盘。")
print("- 春节、国庆、五一前后建议单独标记业务事件，不要只按普通月份比较。")


## 10. 下一步可以补充的业务字段

如果后续要更准确地区分“真实冲量”和“日历结构”，建议补充这些字段：

- 销售政策、考核截止日、价格/返利变动日期。
- 渠道库存或压货相关指标。
- 医院/客户覆盖数量变化，区分 `qty` 是由单客采购变大，还是客户数变多。
- 大促、招采、供货异常、节前停发货等事件标签。

当前 notebook 先把日历和贡献结构讲清楚，方便业务和产品先对齐问题定义。
